# EEG Preprocessing Pipeline – Iowa Gambling Task

**Dataset**: Chávez-Sánchez et al., *Scientific Data* (2026)  
**Obiettivo**: Classificazione decisioni vantaggiose (C/D) vs svantaggiose (A/B) da segnale EEG  

---

## Setup

In [ ]:
# Installa dipendenze se necessario
# !pip install mne numpy pandas scipy matplotlib seaborn

import sys
sys.path.insert(0, '../src')  # aggiunge src/ al path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

from eeg_preprocessing import (
    discover_subjects,
    load_eeg_csv,
    load_igt_csv,
    build_mne_raw,
    set_reference,
    apply_filters,
    run_ica,
    normalize_signal,
    sync_eeg_igt,
    extract_epochs,
    save_subject_output,
    run_full_pipeline,
    plot_raw_vs_clean,
    plot_psd,
    plot_epoch_mean,
    CH_NAMES, SFREQ, TMIN, TMAX, DECK_LABEL_MAP
)

print('Librerie caricate correttamente ✓')
print(f'MNE version: {mne.__version__}')

---
## 1. Configurazione percorsi

Scarica il dataset da Mendeley Data: https://data.mendeley.com/datasets/2pw2m39yct/2

Struttura attesa:
```
dataset_root/
├── participants.csv
├── s-01/
│   ├── s-01_eeg.csv
│   └── s-01_igt.csv
├── s-02/
│   └── ...
```

In [ ]:
DATASET_ROOT = '../data/igt_eeg_dataset'   # <-- MODIFICA con il percorso reale
OUTPUT_DIR   = '../output'

# Scopri soggetti disponibili
subjects = discover_subjects(DATASET_ROOT)
print(f'Soggetti trovati: {len(subjects)}')
print('Primi 5:', [s.split('/')[-1] for s in subjects[:5]])

---
## 2. Pipeline su un singolo soggetto (step-by-step)

In [ ]:
# Lavora sul primo soggetto per ispezione dettagliata
subject_dir = subjects[0]
subject_id  = subject_dir.split('/')[-1]
print(f'Soggetto: {subject_id}')

# Caricamento dati
eeg_data = load_eeg_csv(subject_dir, subject_id)
igt_df   = load_igt_csv(subject_dir, subject_id)

print(f'\nEEG shape: {eeg_data.shape}')
print(f'Durata registrazione: {eeg_data.shape[0]/SFREQ:.1f}s')
print(f'\nIGT trial: {len(igt_df)}')
print(igt_df.head())

In [ ]:
# Distribuzione deck nel dataset
if 'DECK' in igt_df.columns:
    deck_counts = igt_df['DECK'].value_counts().sort_index()
    fig, ax = plt.subplots(figsize=(6, 3))
    colors = ['firebrick', 'tomato', 'steelblue', 'royalblue']
    deck_counts.plot(kind='bar', ax=ax, color=colors, edgecolor='k')
    ax.set_title(f'Distribuzione scelte per deck – {subject_id}')
    ax.set_xlabel('Deck'); ax.set_ylabel('N. scelte')
    for i, (deck, cnt) in enumerate(deck_counts.items()):
        lbl = 'Risky' if deck in ('A', 'B') else 'Safe'
        ax.text(i, cnt + 0.5, lbl, ha='center', fontsize=8, color='gray')
    plt.tight_layout(); plt.show()

In [ ]:
# Step 1: Crea oggetto MNE Raw
raw = build_mne_raw(eeg_data)
print(raw.info)

In [ ]:
# Step 2: PSD del segnale grezzo
fig = raw.plot_psd(fmax=80, average=True, show=False)
fig.suptitle(f'PSD segnale raw – {subject_id}', fontsize=11)
plt.show()

In [ ]:
# Step 3: Re-referencing + filtri
raw_original = raw.copy()   # salva copia originale per confronto
raw = set_reference(raw)
raw = apply_filters(raw)

# Confronto PSD prima/dopo
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
raw_original.plot_psd(fmax=80, average=True, axes=axes[0], show=False)
axes[0].set_title('PSD – Segnale raw')
raw.plot_psd(fmax=80, average=True, axes=axes[1], show=False)
axes[1].set_title('PSD – Dopo filtro bandpass 0.5–70 Hz + notch')
plt.tight_layout(); plt.show()

In [ ]:
# Step 4: ICA
raw, ica = run_ica(raw)

# Visualizza componenti ICA
print(f'Componenti escluse: {ica.exclude}')
if ica.exclude:
    fig = ica.plot_components(picks=ica.exclude[:min(5, len(ica.exclude))], show=False)
    plt.suptitle('Componenti ICA escluse (artefatti oculari)', fontsize=11)
    plt.show()

In [ ]:
# Step 5: Normalizzazione z-score
raw = normalize_signal(raw)

# Verifica: distribuzione ampiezza per canale
data = raw.get_data() * 1e6  # µV
print(f'Media canali (post-norm): {data.mean(axis=1).mean():.4f} (atteso ≈ 0)')
print(f'Std canali (post-norm):   {data.std(axis=1).mean():.4f} (atteso ≈ 1)')

In [ ]:
# Confronto visivo: segnale raw vs preprocessato (Fz, canale frontale)
ch = 'Fz'
if ch in raw.ch_names:
    ch_idx = raw.ch_names.index(ch)
    n_pts = int(5 * SFREQ)  # 5 secondi
    times = np.arange(n_pts) / SFREQ

    fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
    axes[0].plot(times, raw_original.get_data()[ch_idx, :n_pts]*1e6,
                 lw=0.8, color='steelblue')
    axes[0].set_title(f'{ch} – Segnale raw'); axes[0].set_ylabel('µV')

    axes[1].plot(times, raw.get_data()[ch_idx, :n_pts],
                 lw=0.8, color='darkorange')
    axes[1].set_title(f'{ch} – Dopo preprocessing (filtro + ICA + z-score)')
    axes[1].set_ylabel('z-score'); axes[1].set_xlabel('Tempo (s)')

    plt.tight_layout(); plt.show()

---
## 3. Sincronizzazione EEG–IGT ed estrazione epoche

In [ ]:
# Sincronizzazione
n_samples = eeg_data.shape[0]
sync_df = sync_eeg_igt(igt_df, n_eeg_samples=n_samples)

print('\nPrimi trial sincronizzati:')
print(sync_df[['TRIAL', 'DECK', 'EEG_SAMPLE_INT', 'EEG_TIME_S', 'LABEL', 'VALID']].head(10))

In [ ]:
# Estrazione epoche [-2s, 0s]
epochs, labels, sample_indices = extract_epochs(raw, sync_df)

X = epochs.get_data()  # (n_epochs, n_channels, n_timepoints)
print(f'\nShape epoche X: {X.shape}')
print(f'  n_epochs      = {X.shape[0]}')
print(f'  n_channels    = {X.shape[1]}')
print(f'  n_timepoints  = {X.shape[2]} ({X.shape[2]/SFREQ:.1f}s a {SFREQ}Hz)')
print(f'\nLabel distribution:')
unique, counts = np.unique(labels, return_counts=True)
for u, c in zip(unique, counts):
    print(f'  {u} ({["disadvantageous","advantageous"][u]}): {c} trial')

In [ ]:
# Visualizzazione ERP medio per classe
channels_to_plot = [c for c in ['Fz', 'Cz', 'F3', 'F4'] if c in epochs.ch_names]
times = epochs.times

fig, axes = plt.subplots(1, len(channels_to_plot),
                          figsize=(4*len(channels_to_plot), 4))
if len(channels_to_plot) == 1:
    axes = [axes]

for i, ch in enumerate(channels_to_plot):
    ch_idx = epochs.ch_names.index(ch)
    for label_id, label_name, color in [
        (1, 'disadvantageous', 'firebrick'),
        (2, 'advantageous',    'steelblue')
    ]:
        ep = epochs[label_name].get_data()[:, ch_idx, :]
        mean_ep = ep.mean(axis=0)
        sem_ep  = ep.std(axis=0) / np.sqrt(len(ep))
        axes[i].plot(times, mean_ep, lw=1.5, color=color,
                     label=f'{label_name} (n={len(ep)})')
        axes[i].fill_between(times, mean_ep-sem_ep, mean_ep+sem_ep,
                              alpha=0.2, color=color)
    axes[i].axvline(0, color='k', lw=1, ls='--')
    axes[i].set_title(ch); axes[i].set_xlabel('Tempo (s)')
    axes[i].set_ylabel('Ampiezza (z-score)'); axes[i].legend(fontsize=7)

fig.suptitle(f'ERP medio – {subject_id} | finestra [{TMIN}s, {TMAX}s]', fontsize=12)
plt.tight_layout(); plt.show()

---
## 4. Salvataggio output

In [ ]:
paths = save_subject_output(subject_id, epochs, labels, sample_indices, OUTPUT_DIR)

# Verifica
X_loaded = np.load(paths['epochs'])
y_loaded = np.load(paths['labels'])
print(f'File caricato correttamente:')
print(f'  X.shape = {X_loaded.shape}  (n_epochs, n_channels, n_timepoints)')
print(f'  y.shape = {y_loaded.shape}  (n_epochs,)')

---
## 5. Pipeline completa su tutti i soggetti

⚠️ Questa cella elabora tutti i soggetti – può richiedere diversi minuti.

In [ ]:
summary = run_full_pipeline(
    dataset_root=DATASET_ROOT,
    output_dir=OUTPUT_DIR,
    save_figures=True,
    subject_limit=None   # None = tutti i soggetti; metti un int per debug
)
summary

In [ ]:
# Statistiche dataset finale
print('=== Dataset summary ===')
print(f"Totale soggetti elaborati: {len(summary)}")
print(f"Totale epoche: {summary['n_epochs'].sum()}")
print(f"  Advantageous:    {summary['n_advantageous'].sum()}")
print(f"  Disadvantageous: {summary['n_disadvantageous'].sum()}")
print(f"\nEpoche per soggetto (media ± std):")
print(f"  {summary['n_epochs'].mean():.1f} ± {summary['n_epochs'].std():.1f}")

# Visualizza distribuzione epoche
fig, ax = plt.subplots(figsize=(10, 4))
x = range(len(summary))
ax.bar(x, summary['n_advantageous'],     label='Advantageous (C/D)',    color='steelblue',  alpha=0.8)
ax.bar(x, summary['n_disadvantageous'],  label='Disadvantageous (A/B)', color='firebrick',  alpha=0.8,
       bottom=summary['n_advantageous'])
ax.set_xticks(list(x))
ax.set_xticklabels(summary['subject_id'], rotation=45, ha='right', fontsize=8)
ax.set_xlabel('Soggetto'); ax.set_ylabel('N. epoche')
ax.set_title('Distribuzione epoche per soggetto')
ax.legend()
plt.tight_layout(); plt.show()

---
## 6. Caricamento dataset per ML (output finale)

Gli array caricati qui sono pronti per essere passati a un classificatore.

In [ ]:
from pathlib import Path

epochs_dir = Path(OUTPUT_DIR) / 'epochs'

all_X, all_y, all_subjects = [], [], []

for label_file in sorted(epochs_dir.glob('*_labels.npy')):
    sid = label_file.stem.replace('_labels', '')
    X_s = np.load(epochs_dir / f'{sid}_epochs.npy')  # (n_ep, n_ch, n_t)
    y_s = np.load(label_file)
    all_X.append(X_s)
    all_y.append(y_s)
    all_subjects.extend([sid] * len(y_s))

X_all = np.concatenate(all_X, axis=0)  # (N_total, 21, 512)
y_all = np.concatenate(all_y, axis=0)  # (N_total,)
subjects_all = np.array(all_subjects)

print(f'Dataset ML completo:')
print(f'  X shape: {X_all.shape}  (trials × channels × timepoints)')
print(f'  y shape: {y_all.shape}')
print(f'  Classi: {np.unique(y_all, return_counts=True)}')
print(f'  Soggetti unici: {len(np.unique(subjects_all))}')
print('\nX_all e y_all sono pronti per cross-validation / classificazione!')